# MathSLM v2 - Hardware-Agnostic Diagnostic Experiment Matrix

This notebook executes the **MathSLM v2 Diagnostic Controlled Matrix** on Google Colab GPU (NVIDIA T4 / V100 / A100) or local CPU/GPU environments.

### Objectives:
1. **Environment Setup & CUDA Verification**: Detect GPU, VRAM, and PyTorch CUDA support.
2. **Tiny Sanity Test**: Verify model fitting and deterministic extraction on a tiny dataset.
3. **4-Way Controlled Matrix (Exp-A, Exp-B, Exp-C, Exp-D)**: Isolate whether 0% accuracy is driven by undertraining, capacity limits, or reasoning loss dilution.
4. **Persistent Artifact Storage**: Mount Google Drive and save experiment logs and checkpoints.

In [ ]:
# Step 1: Mount Google Drive & Setup Repository Working Directory
import os, sys

# 1. Mount Google Drive (if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/MathSLM_v2'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Google Drive mounted successfully at', DRIVE_DIR)
except Exception as e:
    print('Running locally or Google Drive not mounted:', e)

# 2. Auto-detect and set working directory to repository root
repo_name = 'KLH-csit-2026-2420090008-mathSlm'
target_path = f'/content/{repo_name}'

if os.path.exists(target_path):
    os.chdir(target_path)
elif not os.path.exists('data/generate_synthetic_v2.py') and os.path.exists('/content'):
    print(f'Cloning {repo_name} repository...')
    !git clone https://github.com/Ankitt-02/KLH-csit-2026-2420090008-mathSlm.git
    if os.path.exists(target_path):
        os.chdir(target_path)

print('✅ Current Working Directory:', os.getcwd())
if os.path.exists('data/generate_synthetic_v2.py'):
    print('✅ Repository files verified successfully!')
else:
    print('⚠️ Warning: data/generate_synthetic_v2.py not found in current working directory!')


In [ ]:
# Step 2: Install dependencies and verify PyTorch CUDA environment
!pip install --quiet torch tokenizers datasets pandas pyyaml tqdm sympy

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('GPU Count:', torch.cuda.device_count())
    print('Total VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('CUDA is not available. Falling back to CPU.')

In [ ]:
# Step 3: Generate SymPy-validated disjoint synthetic dataset
import os
!PYTHONUNBUFFERED=1 python3 data/generate_synthetic_v2.py


In [ ]:
# Step 4: Run Tiny Sanity Test
# Verifies pipeline and model capacity to fit small dataset
import os
!PYTHONUNBUFFERED=1 python3 scripts/run_tiny_sanity.py


In [ ]:
# Step 5: Execute 4-Way Controlled Diagnostic Matrix (Exp-A, Exp-B, Exp-C, Exp-D)
# Runs Exp-A (7.34M Direct 5ep), Exp-B (30M Direct 5ep), Exp-C (7.34M Reasoning 5ep), Exp-D (7.34M Reasoning 1ep)
import os
!PYTHONUNBUFFERED=1 python3 scripts/run_controlled_matrix.py auto


In [ ]:
# Step 6: Copy experiment artifacts & logs to Google Drive if mounted
import shutil
if os.path.exists('/content/drive/MyDrive/MathSLM_v2'):
    print('Backing up checkpoints and logs to Google Drive...')
    shutil.copytree('logs', '/content/drive/MyDrive/MathSLM_v2/logs', dirs_exist_ok=True)
    shutil.copytree('checkpoints', '/content/drive/MyDrive/MathSLM_v2/checkpoints', dirs_exist_ok=True)
    print('Backup complete!')